<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Lesson 2 精华 — Research Agent

**一句话定位**:**单 agent + Tavily 搜索 + `think_tool`** 构成一个能 **不无限搜下去** 的研究循环 —— 关键不是模型多强,是 **prompt 怎么写 + context 怎么管**。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**🤔 为啥研究任务最适合用 agent?**

研究是 **开放性任务** —— 最佳策略 **事先不可知**:

| 请求类型 | 适合的策略 |
|---------|----------|
| 「对比 A 和 B」 | 各搜一遍 → 综合对比 |
| 「列出 top 候选人」 | 开放性搜索 → 综合 + 排序 |
| 「研究最新进展」 | 时间过滤 + 多源交叉验证 |

**workflow 不行**(路径不确定),**agent 才行**(运行时根据结果决定下一步)。

但 agent 有个 **致命缺陷**:**容易无限搜下去** —— 这是本节解决的核心问题。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🏗️ Agent 架构 — 4 个节点

</div>

```
Research Brief
      │
      ▼
┌──────────────────┐
│ ① LLM Decision   │ ◀──────┐
│   决定调工具还是  │        │
│   给最终回复      │        │
└────┬─────────────┘        │
     │                       │
  ┌──┴────────┐              │
  │ 有 tool_calls? │              │
  └──┬──────┬──┘              │
 YES │      │ NO              │
     ▼      ▼                  │
┌──────────┐  ┌──────────────┐│
│ ② Tool   │  │ ③ Compress   ││
│  Execute │  │  Research    ││
│ (Tavily/ │  │  (LLM 总结)  ││
│ think)   │  └──────┬───────┘│
└────┬─────┘          │        │
     │ ToolMessage    ▼        │
     └───────────► END         │
                  (返回 notes) │
                               │
          └────回到 ①───────────┘
```

| 节点 | 干啥 | 关键技术 |
|------|------|---------|
| **① LLM Decision** | 看 state,决定调工具 or 终止 | `bind_tools` + `tool_choice` |
| **② Tool Execution** | 执行 `tavily_search` 或 `think_tool` | LangChain tools + Tavily SDK |
| **③ Research Compression** | 把所有 search 结果 **压成 notes** | LLM + 仔细的 prompt |
| **④ Routing Logic** | 条件边 | 看 last message 有没有 tool_calls |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛑 防止「搜索失控」— 3 个 Prompt 技巧

</div>

**最大威胁**:agent **永不满意**,不断搜索,搜到 20 次还在搜。

```
❌ 失控的 agent:
"best coffee shops SF" → "Saint Frank details" → "Sightglass details" 
→ "Ritual details" → "Blue Bottle details" → ... (20+ 次搜索)
→ context 爆炸,质量反而下降
```

**3 个对治技巧**:

### 🅰️ 技巧 ① — 像 agent 一样思考

把 agent 当 **新员工** 教:

| 指令 | 意图 |
|------|------|
| **仔细读问题** | 锁定目标 |
| **从宽泛搜索开始** | 别一开始就钻细节 |
| **每次搜完暂停评估** | 强制「停一下」 |
| **逐渐细化** | **以填补已知 gap 为目的搜** |

### 🅱️ 技巧 ② — 具体启发(硬限制)

| 限制 | 数值 |
|------|------|
| 简单 query 上限 | **2-3 次搜索** |
| 复杂 query 上限 | **5 次搜索** |
| 绝对上限 | **5 次硬停** |
| 「**能自信回答就停**」 | 不追求完美 |

### 🅲 技巧 ③ — 用 `think_tool` 展示思考

每次搜完调一次 [`think_tool`](https://www.anthropic.com/engineering/claude-think-tool):

| 自问 | 决策 |
|------|------|
| 关键信息是什么? | 总结 |
| 还缺什么? | 评估 |
| 够回答了吗? | 决定终止 |
| 该再搜还是给答案? | 走向下一步 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 三个技巧的化学反应**

**没有技巧时**:

```
search("X") → search("X detail 1") → search("X detail 2") → ... 20+ 次
```

**加了 3 技巧后**:

```
search("X") → think_tool(分析结果) → search("补充查 Y") → think_tool(评估完整性) → 给答案
总共 3-5 次搜索
```

**核心洞察**:「**像一个时间有限的人类研究员一样思考**」—— 这避免了「**搜索失控**」(agent 无限搜)的问题。

→ `think_tool` 是 **信号型工具**:它没什么副作用,价值是 **让 agent 自言自语**,把推理过程写进 messages,LLM 下一轮看到自己刚才的思考。

📎 工具类型详见 [`0_c_⭐️_工具类型大全`(隔壁项目)](../../project-002-ambiant-agents/notebooks/0_c_⭐️_工具类型大全.ipynb)

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🌐 Tavily 集成 + 替代方案

</div>

**[Tavily](https://docs.tavily.com/sdk/python/reference)** — 为 AI 应用优化的搜索引擎,免费额度足。

| 能力 | 说明 |
|------|------|
| **实时搜索** | live web results |
| **内容提取 + 摘要** | 自动 clean 噪音 |
| **域名过滤** | 控制可信源 |
| **异步支持** | 性能 |

### 🔄 2026 替代方案对比

| 工具 | 特点 | 适合 |
|------|------|------|
| **Tavily** | search-first,带 AI 摘要 | 通用研究 |
| **[Jina Reader](https://jina.ai/reader/)** | URL 前加 `https://r.jina.ai/` 就转 LLM 友好文本 | **最简单**,只要从单个 URL 取内容 |
| **[Firecrawl](https://www.firecrawl.dev/)** | 全能爬虫,JS 渲染 / sitemap 爬取 | 复杂网站、批量抓取 |
| **Tavily Extract** | 专门的提取 endpoint | 已知 URL 列表,只要提内容 |

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026 实战建议**

- **新手 / 通用** → 直接用 Tavily
- **要全部内容(不只是摘要)** → Jina Reader(超简单)
- **大规模爬 + JS 渲染** → Firecrawl
- **生产场景** → 把多种工具 **作为可选 tool 给 agent**,让它根据需要选

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 Context Engineering — 本节最深的内容

</div>

Research agent 跑久了,**context 会爆炸**(每次 search 都几千字)。本节用 **两层压缩** 应对:

### 📦 第一层 — 网页内容摘要(`summarize_webpage_content`)

**问题**:原始网页含大量噪音(导航栏、广告、模板内容)。

**做法**:用 LLM + structured output **提取关键信息和摘录**:

| 操作 | 目的 |
|------|------|
| 提取关键信息 | 主线 |
| 保留相关摘录 | 引用证据 |
| 过滤无关内容 | 减噪音 |
| 保留来源归属 | 可核查 |

### 📦 第二层 — 研究结果压缩(`compress_research`)

**问题**:多次 search 后,messages 列表里有很多 ToolMessage,**整体 context 太大**。

**做法**:跑完搜索后,**LLM 把所有发现综合成 cohesive 笔记**:

| 输出 | 用途 |
|------|------|
| `notes`(压缩摘要) | 后续 LLM 调用 |
| `raw_notes`(原始 trail) | 详细分析 / 报告引用 |

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 压缩是双刃剑 — 容易丢信息**

[Manus 团队](https://manus.im/blog/Context-Engineering-for-AI-Agents-Lessons-from-Building-Manus) 和 [Cognition](https://cognition.ai/blog/dont-build-multi-agents) 都警告过:**压缩有风险**。

常见失败:

- 压缩 LLM 看长 trajectory 时,**遗忘原始任务** → 输出泛泛而谈
- 关键细节(数字、引用、专有名词)**被概括掉**
- 多次压缩后,**信息层层丢失**(像「无损 → 有损 → 极度有损」)

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 怎么 *小心* 做压缩 — 本节的关键技巧**

我们用 LLM 做压缩,但有 **3 重保护**:

### ① System Prompt 写得严格

在压缩 LLM 的 system prompt 里 **明确**:**「**保留所有跟原任务相关的细节,不要泛化**」**。

### ② 加 `compress_research_human_message`

长 trajectory 可能让 LLM **遗忘任务指令**([context 失败常见模式](https://www.dbreunig.com/2025/06/22/how-contexts-fail-and-how-to-fix-them.html))。

**对治**:在 trajectory 之后 **再补一条 human message**,明确强调:

- 显式重述原始研究主题
- 提醒 **保留所有相关信息**
- 强调全面 findings 对最终报告的重要性
- **防止任务漂移**

**类比**:像考试前 **再读一遍题目**,防止跑题。

### ③ 显式设大 `max_tokens`

压缩输出可能很长。**不显式设 `max_tokens`** → 默认值可能很小 → **输出被截断**(经常看到「**Sextant Coffee Ro**...」突然断尾就是这个问题)。

| 模型 | 上限 | 注意 |
|------|------|------|
| Claude Sonnet 4 | 64k | SDK 默认 1024,**必须显式调大** |
| GPT-4.1 | 33k | 同样要显式 |

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔍 InjectedToolArg — LLM 看不到的隐藏参数

</div>

Tavily 工具的某些参数(如 `topic`、`search_depth`)标了 [`InjectedToolArg`](https://python.langchain.com/api_reference/core/tools/langchain_core.tools.base.InjectedToolArg.html):

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from typing import Annotated
from langchain_core.tools import InjectedToolArg

@tool
def tavily_search(
    query: str,                                                          # LLM 能看到
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">max_results: Annotated[int, InjectedToolArg] = 1</span>,                # LLM 看不到
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">topic: Annotated[Literal["general", "news"], InjectedToolArg] = "general"</span>,
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">config: Annotated[RunnableConfig, InjectedToolArg] = None</span>,
) -> str:
    ...
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 `InjectedToolArg` 的作用**

标了这个的参数 **不会出现在给 LLM 的 schema 里**,而是 **在执行时由框架注入**。

**好处**:

| 优点 | 例子 |
|------|------|
| ✅ 注入 runtime 值 | api_key、user_id、tracing config |
| ✅ 减负 LLM | 不让它管不该管的参数 |
| ✅ 避免出错 | LLM 不会乱填 `max_results=999` 把账单打爆 |
| ✅ 集中管理 | 调参在代码层面,不需要改 prompt |

**类比**:像 web 框架的 dependency injection —— **业务函数声明它需要 db / logger / config,框架运行时注入**,业务代码不用关心怎么拿。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧪 Eval — 测 agent 的「停 vs 继续」决策

</div>

**两种失败模式**:

| 失败模式 | 表现 | 后果 |
|---------|------|------|
| **Early Termination**(早停) | 任务没完成就停了 | **答案太浅** |
| **Prolonged Looping**(死循环) | 永不满意 | **token 浪费 + context 污染** |

本节用 **简易 eval dataset** 测试 agent 决策:

| 测试 | 期望 |
|------|------|
| 例 1:信息不全的搜索 | **应该继续** |
| 例 2:信息齐全的搜索 | **应该停下** |

**evaluator 用启发式**(不用 LLM-as-judge):

```python
if last_message.tool_calls:
    decision = "continue"
else:
    decision = "stop"
assert decision == expected_next_step
```

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**🎁 一个学到的好 trick**

```python
agent.nodes["node_name"].invoke(...)
```

可以 **单独测某一个 node 的行为**,不用跑完整流程!这对 **debug + unit test 非常有用** —— 不用每次都跑完整 graph(慢且贵)。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🎭 类比 — Research Agent 像什么?

</div>

### 🅰️ 类比:**截止日期前 1 小时的学生**

好学生跟坏学生的区别 **不是搜的能力**,是 **何时停**:

| 坏学生 | 好学生 |
|--------|--------|
| 搜了 20 个网页还在搜「再多看一眼」 | 搜 3-5 个就开始写 |
| 拿到大量内容不会取舍 | **每搜一次就反思** 一下还缺啥 |
| 截止时间到了交不出来 | **能自信回答就停**,不追求完美 |

→ 我们 prompt 教 agent 当 **「**好学生**」**。

### 🅱️ 类比:**记者写稿**

好记者:

- **先列大纲**(知道要回答什么)
- **查资料填空白**(不是无目的乱查)
- **每查一段反思**(还缺哪个角度)
- **够了就停笔**(deadline 比完美重要)

**`think_tool`** = 记者的「**笔记本**」,把思考写下来,**避免脑子里乱**。

### 🆚 对比:**没有 context engineering 的 agent**

| 没压缩 | 有两层压缩 |
|--------|----------|
| 5 次 search = 几万 tokens | 5 次 search → notes 几千字 |
| 后续 LLM 每次重读所有原始内容 | 后续看摘要 |
| 成本暴增 | 成本可控 |
| context rot 严重 | 信息聚焦 |

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 5 个最容易踩的坑**

1. **❌ 不在 prompt 里设硬限制** → agent 搜 20+ 次,token 爆炸

2. **❌ 没加 `think_tool`** → agent 直接搜 → 直接答,没有反思机制,质量不稳定

3. **❌ `max_tokens` 用 SDK 默认** → 压缩输出被截断,「Sextant Coffee Ro...」式断尾

4. **❌ 压缩 LLM 的 prompt 不强调「保留细节」** → 输出泛泛而谈,后续报告基于残缺信息

5. **❌ 把 runtime config / api_key 直接放进工具签名** → LLM 会尝试乱填这些字段,**永远用 `InjectedToolArg`**

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**Research Agent 的核心 = 「**ReAct 循环 + Context Engineering + 防失控 prompt**」**:

- **架构**:4 节点(decision / execute / compress / route)
- **3 个 prompt 技巧** 防止「搜索失控」:像 agent 一样思考 + 硬限制 + `think_tool`
- **两层压缩** 控 context:网页摘要 + 研究压缩
- **小心做压缩** 的 3 重保护:严格 prompt + 任务复述 + 显式 max_tokens
- **`InjectedToolArg`** 把 LLM 不该管的参数藏起来

### 🎯 学完这节,你应该能回答:

1. ✅ **为啥研究任务适合 agent 不适合 workflow?**(运行时才知道怎么走)
2. ✅ **怎么防止 agent 无限搜?**(prompt 3 技巧 + hard limits)
3. ✅ **`think_tool` 是什么类型工具?**(信号型,价值在调用本身 / 让 agent 自言自语)
4. ✅ **为啥要做两层压缩?**(网页噪音 + 多 search 累积 → 双层减负)
5. ✅ **压缩有什么风险?怎么对治?**(丢信息 / 任务漂移;严格 prompt + 任务复述 + max_tokens)
6. ✅ **`InjectedToolArg` 干嘛的?**(藏起 LLM 不该管的参数,如 api_key)

📎 配套阅读:[`2_research_agent.ipynb` 主课](./2_research_agent.ipynb) · 下一节 [`3_z_⭐️_本节精华_MCP.ipynb`](./3_z_⭐️_本节精华_MCP.ipynb)

</div>